# 🔁 Transfer learning: Steam žanras iš viršelio su MobileNetV2
**Užduotis 1.5** - Transfer learning Colab'ų perdarymas su savo duomenimis.

## Kas yra transfer learning?
`MobileNetV2` jau išmokytas ties **ImageNet** (1,4 mln. nuotraukų, 1000 klasių) ir moka atpažinti kraštus, tekstūras, objektus. Mes **nemokome iš nulio**, o pakartotinai panaudojame šias žinias:

1. **Feature extraction (išskyrimas)**: užšaldome visą MobileNetV2 (`trainable=False`) ir apmokome tik naują viršutinį klasifikatorių.
2. **Fine-tuning (smulkus derinimas)**: atšaldome paskutinius ~30 pagrindinio tinklo sluoksnių ir tęsiame mokymą su **labai mažu** mokymosi greičiu (1e-5), kad nesugadintume jau išmoktų požymių.
3. **Variantas B (embeddings)**: iš MobileNetV2 paimame požymių vektorius kaip „įterpinius“ ir ant jų apmokome paprastą logistinę regresiją, o `t-SNE` atvaizduojame, kaip žanrai išsidėsto erdvėje.

Svarbu: kiekvienas ImageNet modelis turi savo **`preprocess_input`** (MobileNetV2 tikisi reikšmių [-1, 1]), todėl jo negalima keisti į paprastą `/255`.

**Prieš paleidžiant:** `Runtime → Change runtime type → T4 GPU`. Jei jau paleidote `04_cnn`, duomenys bus perpanaudoti (tas pats `/content/steam_genres`).

In [ ]:
import os, time, random, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
SEED = 42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED); sns.set_theme(style="whitegrid")

GENRES = ["Racing", "Strategy", "Sports", "RPG"]   # <-- galima keisti (SteamSpy žanrai: Action, Adventure, Casual, Indie, RPG, Simulation, Strategy, Racing, Sports ...)
PER_GENRE = 250
ROOT = "/content/steam_genres"
HEADER = "https://cdn.cloudflare.steamstatic.com/steam/apps/{}/header.jpg"

def top_apps(genre, k):
    j = requests.get("https://steamspy.com/api.php", params={"request": "genre", "genre": genre}, timeout=60).json()
    apps = sorted(j.values(), key=lambda a: a["positive"] + a["negative"], reverse=True)   # populiariausi turi tvarkingus viršelius
    return [(a["appid"], a["name"]) for a in apps[:k * 2]]

def download(args):
    appid, genre = args
    out = f"{ROOT}/{genre}/{appid}.jpg"
    if os.path.exists(out): return out
    try:
        r = requests.get(HEADER.format(appid), timeout=15)
        if r.status_code == 200 and len(r.content) > 5000:
            open(out, "wb").write(r.content); return out
    except Exception:
        pass
    return None

if not os.path.isdir(ROOT):
    cand = {}
    for g in GENRES:
        cand[g] = top_apps(g, PER_GENRE); time.sleep(2)
    # pašaliname žaidimus, priklausančius keliems pasirinktiems žanrams (kad klasės nesidubliuotų)
    cnt = {}
    for g, lst in cand.items():
        for appid, _ in lst: cnt[appid] = cnt.get(appid, 0) + 1
    jobs = []
    for g, lst in cand.items():
        os.makedirs(f"{ROOT}/{g}", exist_ok=True)
        jobs += [(a, g) for a, _ in [x for x in lst if cnt[x[0]] == 1]]
    with ThreadPoolExecutor(16) as ex: res = list(ex.map(download, jobs))
    # išlyginame klases: palieka tiek vaizdų, kiek yra mažiausioje klasėje (iki PER_GENRE)
    for g in GENRES:
        files = sorted(os.listdir(f"{ROOT}/{g}"))
        for f in files[PER_GENRE:]: os.remove(f"{ROOT}/{g}/{f}")

paths, labels = [], []
for i, g in enumerate(GENRES):
    fs = sorted(os.listdir(f"{ROOT}/{g}")); paths += [f"{ROOT}/{g}/{f}" for f in fs]; labels += [i] * len(fs)
paths, labels = np.array(paths), np.array(labels)
print(pd.Series(labels).map(dict(enumerate(GENRES))).value_counts())

In [ ]:
IMG_SIZE = (128, 256)   # viršeliai ~460x215, išlaikome santykį maždaug 1:2
BATCH = 32; AUTOTUNE = tf.data.AUTOTUNE

p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print("train/val/test:", len(p_tr), len(p_va), len(p_te))

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(p, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y)).map(load, num_parallel_calls=AUTOTUNE).cache()
    if training: ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(p_tr, y_tr, True), make_ds(p_va, y_va), make_ds(p_te, y_te)

imgs, labs = next(iter(train_ds))
plt.figure(figsize=(12, 5))
for i in range(8):
    ax = plt.subplot(2, 4, i + 1); ax.imshow(imgs[i].numpy().astype("uint8")); ax.set_title(GENRES[int(labs[i])]); ax.axis("off")
plt.tight_layout(); plt.show()

## 1. Modelis: MobileNetV2 + naujas klasifikatorius
`include_top=False` pašalina originalų 1000 klasių sluoksnį. Mūsų viršus: `GlobalAveragePooling2D → Dropout → Dense(softmax)`.

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

aug = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomZoom(0.1)], name="augmentation")
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = aug(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)          # training=False svarbu: BatchNorm sluoksniai lieka „inferencijos“ režime
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(GENRES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Mokomų parametrų:", sum(int(np.prod(w.shape)) for w in model.trainable_weights),
      "| iš viso:", model.count_params())

## 2. Fazė 1: feature extraction
Mokosi tik galutinis `Dense` sluoksnis, todėl tai greita ir veikia net su mažu duomenų kiekiu.

In [ ]:
cbs = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)
acc1 = model.evaluate(test_ds, verbose=0)[1]; print(f"Testas po feature extraction: {acc1:.3f}")

## 3. Fazė 2: fine-tuning
Atšaldome paskutinius 30 MobileNetV2 sluoksnių ir **iš naujo kompiliuojame** su mažesniu mokymosi greičiu (kitaip pakeitimai nesuveiks).

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)
acc2 = model.evaluate(test_ds, verbose=0)[1]; print(f"Testas po fine-tuning: {acc2:.3f}")

In [ ]:
H = pd.concat([pd.DataFrame(h1.history), pd.DataFrame(h2.history)], ignore_index=True)
plt.figure(figsize=(8, 4)); plt.plot(H.accuracy, label="train"); plt.plot(H.val_accuracy, label="val")
plt.axvline(len(h1.history["loss"]) - 0.5, color="k", ls="--", label="fine-tuning pradžia"); plt.legend()
plt.title("Transfer learning: tikslumas"); plt.xlabel("epocha"); plt.show()

y_pred = np.concatenate([model.predict(x, verbose=0).argmax(1) for x, _ in test_ds])
y_true = np.concatenate([y.numpy() for _, y in test_ds])
print(classification_report(y_true, y_pred, target_names=GENRES))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=GENRES, yticklabels=GENRES)
plt.xlabel("Prognozė"); plt.ylabel("Tikra"); plt.show()

## 4. Variantas B: įterpiniai + logistinė regresija + t-SNE
Gauname 1280 skaičių vektorių kiekvienam paveikslėliui. Jei žanrai vektorių erdvėje sudaro grupes, tai rodo, kad ImageNet požymiai tinka mūsų užduočiai.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
extractor = tf.keras.Sequential([tf.keras.layers.Input(shape=IMG_SIZE + (3,)),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input), base, tf.keras.layers.GlobalAveragePooling2D()])
emb = lambda ds: np.concatenate([extractor.predict(x, verbose=0) for x, _ in ds])
E_tr, E_te = emb(make_ds(p_tr, y_tr)), emb(test_ds)
clf = LogisticRegression(max_iter=2000).fit(E_tr, y_tr)
acc_emb = clf.score(E_te, y_true); print(f"Įterpiniai + LogReg testo tikslumas: {acc_emb:.3f}")
Z = TSNE(n_components=2, random_state=SEED, perplexity=25).fit_transform(E_te)
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=[GENRES[i] for i in y_true], palette="tab10"); plt.title("t-SNE (testo aibė)"); plt.show()

## 5. Palyginimas ir išvados
Jei `04_cnn` buvo paleistas toje pačioje sesijoje, kintamasis `CNN_TEST_ACC` bus užpildytas; kitu atveju įrašykite reikšmę ranka.

In [ ]:
try: cnn = CNN_TEST_ACC
except NameError: cnn = float("nan")
res = pd.DataFrame({"modelis": ["CNN iš nulio", "MobileNetV2 feature extraction", "MobileNetV2 + fine-tuning", "Įterpiniai + LogReg"],
                    "testo tikslumas": [cnn, acc1, acc2, acc_emb]}).set_index("modelis")
display(res.round(3)); res.plot.barh(legend=False, color="#5c6bc0", title="Testo tikslumas"); plt.show()

* Transfer learning paprastai gerokai pranoksta CNN iš nulio, nes mažos imties problema sprendžiama panaudojant ImageNet žinias.
* Fine-tuning kartais duoda tik nedidelį pagerėjimą arba net permoko, todėl būtinas mažas LR ir stebėjimas pagal `val_accuracy`.
* Apribojimas: viršelis yra marketingas (kaip ir kūrėjų aprašymas!). Tai dera su projekto mintimi: žanrą iš viršelio spėti sunku, o žaidėjų apžvalgos (Gemini notebook'as) parodo, koks žaidimas yra iš tikrųjų.